In [1]:
import os
import sys
import json
import time
import random
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

try:
    import psutil
except Exception:
    psutil = None

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "fine-tuning" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import SPLICING_CLASS_NAMES  # noqa: E402
from models import FoundationModelLoader  # noqa: E402
from splicing_metrics import MetricsComputer  # noqa: E402


# ==========================
# User Config
# ==========================
WINDOW_SIZE = 2000
TEST_CHROMS = {20, 21}

MODEL_NAME = "NucleotideTransformer"
MODEL_ID = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
MAX_LENGTH = 2000
DROPOUT = 0.2
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()

# 1 fold: single train/val split (no CV)
VAL_SIZE = 0.1

EXPERIMENT_NAME = f"NT_finetune_gencode_train_w{WINDOW_SIZE}_fold1"

RESULTS_ROOT = PROJECT_ROOT / "results" / "fine_tuning_nt"
EXPERIMENT_DIR = RESULTS_ROOT / EXPERIMENT_NAME
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
SUMMARY_DIR = RESULTS_ROOT / "summaries"
TELEMETRY_DIR = PROJECT_ROOT / "results" / "pipeline_telemetry"
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "fine_tuning_nt" / EXPERIMENT_NAME / "tensorboard"

for p in [EXPERIMENT_DIR, CHECKPOINT_DIR, SUMMARY_DIR, TELEMETRY_DIR, TENSORBOARD_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RAW_RATIO_TAG = "1_1_raw"
RATIOS = [RAW_RATIO_TAG]


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_chrom_column(df: pd.DataFrame, chrom_col: str = "CHROM") -> pd.DataFrame:
    out = df.copy()
    if out[chrom_col].dtype == "object":
        out[chrom_col] = out[chrom_col].str.replace("chr", "", regex=False)
        out[chrom_col] = out[chrom_col].replace({"X": 23, "Y": 24, "M": 25, "MT": 25})
        out[chrom_col] = pd.to_numeric(out[chrom_col], errors="coerce")
    out = out.dropna(subset=[chrom_col]).copy()
    out[chrom_col] = out[chrom_col].astype(int)
    return out


@dataclass
class SplitData:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    gencode_test_df: pd.DataFrame
    gtex_test_df: pd.DataFrame


def build_splits(window_size: int) -> SplitData:
    gencode_path = PROJECT_ROOT / "gencode_multi_seq_length" / f"gencode{window_size}.csv"
    gtex_path = PROJECT_ROOT / "gtex_multi_seq_length" / f"gtex{window_size}.csv"

    gencode_df = pd.read_csv(gencode_path)
    gtex_df = pd.read_csv(gtex_path)

    gencode_df = normalize_chrom_column(gencode_df)
    gtex_df = normalize_chrom_column(gtex_df)

    gencode_test_df = gencode_df[gencode_df["CHROM"].isin(TEST_CHROMS)].copy()
    gencode_trainval_df = gencode_df[~gencode_df["CHROM"].isin(TEST_CHROMS)].copy()

    gtex_test_df = gtex_df[gtex_df["CHROM"].isin(TEST_CHROMS)].copy()

    train_df, val_df = train_test_split(
        gencode_trainval_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=gencode_trainval_df["Splicing_types"],
    )

    return SplitData(
        train_df=train_df,
        val_df=val_df,
        gencode_test_df=gencode_test_df,
        gtex_test_df=gtex_test_df,
    )


class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], int(self.labels[idx])


class NTSplicingModel(nn.Module):
    def __init__(self, backbone: nn.Module, hidden_size: int, num_classes: int = 3, dropout: float = 0.2):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state

        mask = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
        pooled = (hidden * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits


def build_collate_fn(tokenizer, max_length: int):
    def collate_fn(batch):
        seqs, labels = zip(*batch)
        enc = tokenizer(
            list(seqs),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
        )
        y = torch.tensor(labels, dtype=torch.long)
        return enc["input_ids"], enc["attention_mask"], y

    return collate_fn


def to_serializable(obj):
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


set_seed(SEED)
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Results dir: {EXPERIMENT_DIR}")

Project root: d:\Splice_FMs_seq_lengths
Device: cuda
Experiment: NT_finetune_gencode_train_w2000_fold1
Results dir: d:\Splice_FMs_seq_lengths\results\fine_tuning_nt\NT_finetune_gencode_train_w2000_fold1


In [2]:
# ==========================
# Load Data + Model
# ==========================
from tqdm.auto import tqdm

split_data = build_splits(WINDOW_SIZE)

print("Data split summary:")
print(f"  Train (GENCODE, chr != 20/21): {len(split_data.train_df):,}")
print(f"  Val   (GENCODE, chr != 20/21): {len(split_data.val_df):,}")
print(f"  Test  (GENCODE, chr 20/21):    {len(split_data.gencode_test_df):,}")
print(f"  Test  (GTEx, chr 20/21):       {len(split_data.gtex_test_df):,}")

loader = FoundationModelLoader(device=DEVICE)
backbone, tokenizer = loader.load_model_by_name(MODEL_NAME, MODEL_ID)

hidden_size = getattr(backbone.config, "hidden_size", 768)
model = NTSplicingModel(backbone=backbone, hidden_size=hidden_size, num_classes=3, dropout=DROPOUT).to(DEVICE)

if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        model.backbone.resize_token_embeddings(len(tokenizer))

collate_fn = build_collate_fn(tokenizer=tokenizer, max_length=MAX_LENGTH)

train_dataset = SequenceDataset(
    split_data.train_df["sequence"].tolist(),
    split_data.train_df["Splicing_types"].astype(int).tolist(),
)
val_dataset = SequenceDataset(
    split_data.val_df["sequence"].tolist(),
    split_data.val_df["Splicing_types"].astype(int).tolist(),
)
gencode_test_dataset = SequenceDataset(
    split_data.gencode_test_df["sequence"].tolist(),
    split_data.gencode_test_df["Splicing_types"].astype(int).tolist(),
)
gtex_test_dataset = SequenceDataset(
    split_data.gtex_test_df["sequence"].tolist(),
    split_data.gtex_test_df["Splicing_types"].astype(int).tolist(),
)

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
gencode_test_loader = DataLoader(gencode_test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
gtex_test_loader = DataLoader(gtex_test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)


def run_eval(eval_model, data_loader):
    eval_model.eval()
    total_loss = 0.0
    all_y = []
    all_pred = []
    all_prob = []

    with torch.no_grad():
        for input_ids, attention_mask, labels in data_loader:
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = eval_model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            all_y.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_prob.append(probs.cpu().numpy())

    y_true = np.array(all_y)
    y_pred = np.array(all_pred)
    y_prob = np.concatenate(all_prob, axis=0) if all_prob else np.zeros((0, 3), dtype=np.float32)

    metrics, cm = MetricsComputer.compute_metrics(y_true, y_pred, y_prob)
    avg_loss = total_loss / max(1, len(data_loader))
    return avg_loss, metrics, cm


best_state = None
best_epoch = -1
best_val_mcc = -1e9
patience_counter = 0
history = []

steps_per_epoch = len(train_loader)
print(f"\nStart fine-tuning (1 fold)...")
print(f"Train steps/epoch: {steps_per_epoch:,} | Epochs: {NUM_EPOCHS}")

global_train_start = time.perf_counter()
epoch_durations = []

for epoch in range(NUM_EPOCHS):
    epoch_start = time.perf_counter()
    model.train()
    train_loss_sum = 0.0

    pbar = tqdm(
        train_loader,
        total=steps_per_epoch,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
        dynamic_ncols=True,
        leave=True,
    )

    for step, (input_ids, attention_mask, labels) in enumerate(pbar, start=1):
        step_start = time.perf_counter()

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=USE_FP16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item()

        step_time = max(time.perf_counter() - step_start, 1e-6)
        remaining_steps = steps_per_epoch - step
        eta_epoch_sec = remaining_steps * step_time

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "avg": f"{(train_loss_sum / step):.4f}",
            "bt": f"{step_time:.2f}s",
            "eta_ep": f"{eta_epoch_sec / 60:.1f}m",
        })

    train_loss = train_loss_sum / max(1, steps_per_epoch)
    val_loss, val_metrics, _ = run_eval(model, val_loader)

    epoch_time = time.perf_counter() - epoch_start
    epoch_durations.append(epoch_time)

    row = {
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "val_mcc": float(val_metrics.get("mcc", 0.0)),
        "val_f1_weighted": float(val_metrics.get("f1_weighted", 0.0)),
        "val_balanced_accuracy": float(val_metrics.get("balanced_accuracy", 0.0)),
        "epoch_time_seconds": float(epoch_time),
    }
    history.append(row)

    avg_epoch_time = float(np.mean(epoch_durations))
    remaining_epochs = NUM_EPOCHS - (epoch + 1)
    eta_total_sec = remaining_epochs * avg_epoch_time
    elapsed_total_sec = time.perf_counter() - global_train_start

    val_mcc = row["val_mcc"]
    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_mcc={val_mcc:.4f} | val_f1w={row['val_f1_weighted']:.4f} | "
        f"epoch_time={epoch_time/60:.1f}m | elapsed={elapsed_total_sec/60:.1f}m | eta_total={eta_total_sec/60:.1f}m"
    )

    if val_mcc > best_val_mcc:
        best_val_mcc = val_mcc
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

if best_state is None:
    raise RuntimeError("Training failed: no best checkpoint state captured.")

model.load_state_dict(best_state)

checkpoint_payload = {
    "fold_idx": 0,
    "fold_number": 1,
    "best_epoch": int(best_epoch + 1),
    "best_metric": "mcc",
    "best_mcc": float(best_val_mcc),
    "experiment_name": EXPERIMENT_NAME,
    "window_size": int(WINDOW_SIZE),
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "num_classes": 3,
    "hyperparameters": {
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "num_epochs": NUM_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "max_length": MAX_LENGTH,
        "dropout": DROPOUT,
        "seed": SEED,
        "val_size": VAL_SIZE,
    },
    "model_state_dict": best_state,
}

best_checkpoint_name = "best_fold1.pt"
best_checkpoint_path = CHECKPOINT_DIR / best_checkpoint_name
torch.save(checkpoint_payload, best_checkpoint_path)

val_loss, val_metrics, val_cm = run_eval(model, val_loader)

results_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "timestamp": datetime.now().isoformat(),
    "training_type": "end_to_end_finetuning",
    "best_fold": {
        "fold_idx": 0,
        "fold_number": 1,
        "best_metric": "mcc",
        "best_mcc": float(best_val_mcc),
        "checkpoint": best_checkpoint_name,
        "checkpoint_path": str(best_checkpoint_path),
    },
    "hyperparameters": checkpoint_payload["hyperparameters"],
    "per_fold_results": {
        "fold_0": {
            "best_epoch": int(best_epoch + 1),
            "best_metric": "mcc",
            "best_mcc": float(best_val_mcc),
            "best_checkpoint": best_checkpoint_name,
            "val_loss": float(val_loss),
            "metrics": {k: float(v) if isinstance(v, (int, float, np.number)) else v for k, v in val_metrics.items()},
            "confusion_matrix": val_cm.tolist(),
        }
    },
    "averaged_metrics": {
        f"{k}_mean": float(v) if isinstance(v, (int, float, np.number)) else v
        for k, v in val_metrics.items()
    },
    "history": history,
}

config_path = EXPERIMENT_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(to_serializable({
        "window_size": WINDOW_SIZE,
        "test_chromosomes": sorted(list(TEST_CHROMS)),
        "model_name": MODEL_NAME,
        "model_id": MODEL_ID,
        **checkpoint_payload["hyperparameters"],
        "device": DEVICE,
    }), f, indent=2)

results_path = EXPERIMENT_DIR / "results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(to_serializable(results_summary), f, indent=2)

print("\nTraining artifacts saved:")
print(f"  results: {results_path}")
print(f"  config: {config_path}")
print(f"  best checkpoint: {best_checkpoint_path}")

Data split summary:
  Train (GENCODE, chr != 20/21): 646,427
  Val   (GENCODE, chr != 20/21): 71,826
  Test  (GENCODE, chr 20/21):    26,364
  Test  (GTEx, chr 20/21):       41,064


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Start fine-tuning (1 fold)...
Train steps/epoch: 80,804 | Epochs: 5


Epoch 1/5:   0%|          | 0/80804 [00:00<?, ?it/s]

...

Epoch 2/5 | train_loss=0.2110 | val_loss=0.2312 | val_mcc=0.9019 | val_f1w=0.9343 | epoch_time=363.5m | elapsed=656.2m | eta_total=984.2m


Epoch 3/5:   0%|          | 0/80804 [00:00<?, ?it/s]

Epoch 3/5 | train_loss=0.1879 | val_loss=0.3287 | val_mcc=0.8741 | val_f1w=0.9136 | epoch_time=400.2m | elapsed=1056.4m | eta_total=704.2m


Epoch 4/5:   0%|          | 0/80804 [00:00<?, ?it/s]

Epoch 4/5 | train_loss=0.1651 | val_loss=0.2694 | val_mcc=0.8968 | val_f1w=0.9312 | epoch_time=391.8m | elapsed=1448.2m | eta_total=362.0m
Early stopping at epoch 4

Training artifacts saved:
  results: d:\Splice_FMs_seq_lengths\results\fine_tuning_nt\NT_finetune_gencode_train_w2000_fold1\results.json
  config: d:\Splice_FMs_seq_lengths\results\fine_tuning_nt\NT_finetune_gencode_train_w2000_fold1\config.json
  best checkpoint: d:\Splice_FMs_seq_lengths\results\fine_tuning_nt\NT_finetune_gencode_train_w2000_fold1\checkpoints\best_fold1.pt


In [3]:
# ==========================
# Inference on GENCODE chr20/21 and GTEx chr20/21 + Telemetry save
# ==========================

def model_param_stats(full_model: nn.Module):
    total = sum(p.numel() for p in full_model.parameters())
    total_mem = sum(p.numel() * p.element_size() for p in full_model.parameters()) / (1024 ** 2)
    head = sum(p.numel() for p in full_model.classifier.parameters())
    head_mem = sum(p.numel() * p.element_size() for p in full_model.classifier.parameters()) / (1024 ** 2)
    backbone = total - head
    backbone_mem = total_mem - head_mem
    return {
        "total_param_count": int(total),
        "total_model_param_memory_mb": float(total_mem),
        "fm_param_count": int(backbone),
        "fm_model_param_memory_mb": float(backbone_mem),
        "mlp_param_count": int(head),
        "mlp_model_param_memory_mb": float(head_mem),
    }


def classifier_gflops_per_sample(full_model: nn.Module):
    flops = 0.0
    for layer in full_model.classifier.modules():
        if isinstance(layer, nn.Linear):
            in_dim = int(layer.in_features)
            out_dim = int(layer.out_features)
            flops += (2.0 * in_dim * out_dim) + out_dim
    return float(flops / 1e9)


def infer_with_runtime(full_model: nn.Module, data_loader: DataLoader):
    full_model.eval()

    cpu_before_mb = np.nan
    cpu_after_mb = np.nan
    if psutil is not None:
        proc = psutil.Process(os.getpid())
        cpu_before_mb = proc.memory_info().rss / (1024 ** 2)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    all_y = []
    all_pred = []
    all_prob = []

    start_time = time.perf_counter()
    with torch.no_grad():
        for input_ids, attention_mask, labels in data_loader:
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = full_model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            all_y.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_prob.append(probs.cpu().numpy())

    infer_seconds = float(time.perf_counter() - start_time)

    gpu_peak_mb = np.nan
    if DEVICE == "cuda":
        gpu_peak_mb = float(torch.cuda.max_memory_allocated() / (1024 ** 2))

    if psutil is not None:
        proc = psutil.Process(os.getpid())
        cpu_after_mb = proc.memory_info().rss / (1024 ** 2)

    y_true = np.array(all_y)
    y_pred = np.array(all_pred)
    y_prob = np.concatenate(all_prob, axis=0) if all_prob else np.zeros((0, 3), dtype=np.float32)

    metrics, cm = MetricsComputer.compute_metrics(y_true, y_pred, y_prob)

    return {
        "num_samples": int(len(y_true)),
        "running_time_seconds": infer_seconds,
        "cpu_ram_delta_mb": float(cpu_after_mb - cpu_before_mb) if psutil is not None else np.nan,
        "gpu_peak_memory_mb": float(gpu_peak_mb),
        "metrics": {k: float(v) if isinstance(v, (int, float, np.number)) else v for k, v in metrics.items()},
        "confusion_matrix": cm.tolist(),
    }


param_info = model_param_stats(model)
mlp_gflops_per_sample = classifier_gflops_per_sample(model)

inference_targets = [
    ("gencode", RAW_RATIO_TAG, gencode_test_loader, PROJECT_ROOT / "gencode_multi_seq_length" / f"gencode{WINDOW_SIZE}.csv"),
    ("gtex", RAW_RATIO_TAG, gtex_test_loader, PROJECT_ROOT / "gtex_multi_seq_length" / f"gtex{WINDOW_SIZE}.csv"),
]

result_rows = []
legacy_inference_results = {}
model_infer_total = 0.0

for dataset_name, ratio_name, data_loader, source_file in inference_targets:
    out = infer_with_runtime(model, data_loader)
    model_infer_total += out["running_time_seconds"]

    row = {
        "timestamp": TIMESTAMP,
        "stage": "mlp_inference",
        "family": MODEL_NAME,
        "model_id": MODEL_ID,
        "model_variant": MODEL_ID,
        "experiment": EXPERIMENT_NAME,
        "window_size": WINDOW_SIZE,
        "dataset": dataset_name,
        "ratio": ratio_name,
        "best_fold_number": 1,
        "checkpoint": best_checkpoint_name,
        "best_checkpoint": str(best_checkpoint_path),
        "embedding_file": str(source_file),
        "running_time_seconds": out["running_time_seconds"],
        "model_inference_seconds_total": np.nan,
        "n_samples": out["num_samples"],
        "param_count": param_info["mlp_param_count"],
        "model_param_memory_mb": param_info["mlp_model_param_memory_mb"],
        "gpu_peak_memory_mb": out["gpu_peak_memory_mb"],
        "cpu_ram_delta_mb": out["cpu_ram_delta_mb"],
        "gflops_per_sample": mlp_gflops_per_sample,
        "gflops_total": float(mlp_gflops_per_sample * out["num_samples"]),
        "confusion_matrix": json.dumps(out["confusion_matrix"]),
        **out["metrics"],
    }
    result_rows.append(row)

for row in result_rows:
    row["model_inference_seconds_total"] = float(model_infer_total)

mlp_df = pd.DataFrame(result_rows)
mlp_long_csv = TELEMETRY_DIR / f"mlp_inference_telemetry_long_{TIMESTAMP}.csv"
mlp_long_json = TELEMETRY_DIR / f"mlp_inference_telemetry_long_{TIMESTAMP}.json"
mlp_df.to_csv(mlp_long_csv, index=False)
mlp_df.to_json(mlp_long_json, orient="records", indent=2)

# Build pipeline-style telemetry compatible with imbalancing notebook schema
pipeline_df = mlp_df.rename(
    columns={
        "running_time_seconds": "mlp_running_time_seconds",
        "param_count": "mlp_param_count",
        "model_param_memory_mb": "mlp_model_param_memory_mb",
        "gpu_peak_memory_mb": "mlp_gpu_peak_memory_mb",
        "cpu_ram_delta_mb": "mlp_cpu_ram_delta_mb",
        "gflops_per_sample": "mlp_gflops_per_sample",
        "gflops_total": "mlp_gflops_total",
    }
).copy()

pipeline_df["fm_running_time_seconds"] = pipeline_df["mlp_running_time_seconds"]
pipeline_df["fm_param_count"] = param_info["fm_param_count"]
pipeline_df["fm_model_param_memory_mb"] = param_info["fm_model_param_memory_mb"]
pipeline_df["fm_gpu_peak_memory_mb"] = pipeline_df["mlp_gpu_peak_memory_mb"]
pipeline_df["fm_cpu_ram_delta_mb"] = pipeline_df["mlp_cpu_ram_delta_mb"]
pipeline_df["fm_gflops_per_sample"] = np.nan
pipeline_df["fm_gflops_total"] = np.nan
pipeline_df["fm_gflops_method"] = "not_measured_end2end"

pipeline_df["pipeline_running_time_seconds"] = pipeline_df["fm_running_time_seconds"] + pipeline_df["mlp_running_time_seconds"]
pipeline_df["pipeline_gflops_total"] = pipeline_df["fm_gflops_total"] + pipeline_df["mlp_gflops_total"]
pipeline_df["pipeline_gpu_peak_memory_mb"] = pipeline_df[["fm_gpu_peak_memory_mb", "mlp_gpu_peak_memory_mb"]].max(axis=1)
pipeline_df["pipeline_cpu_ram_delta_mb"] = pipeline_df["fm_cpu_ram_delta_mb"] + pipeline_df["mlp_cpu_ram_delta_mb"]
pipeline_df["pipeline_param_count_total"] = pipeline_df["fm_param_count"] + pipeline_df["mlp_param_count"]
pipeline_df["pipeline_model_param_memory_mb_total"] = pipeline_df["fm_model_param_memory_mb"] + pipeline_df["mlp_model_param_memory_mb"]

pipe_long_csv = TELEMETRY_DIR / f"pipeline_telemetry_long_{TIMESTAMP}.csv"
pipe_long_json = TELEMETRY_DIR / f"pipeline_telemetry_long_{TIMESTAMP}.json"
pipeline_df.to_csv(pipe_long_csv, index=False)
pipeline_df.to_json(pipe_long_json, orient="records", indent=2)

wide_metrics = [
    "balanced_accuracy",
    "mcc",
    "f1_weighted",
    "pipeline_running_time_seconds",
    "pipeline_gflops_total",
    "pipeline_param_count_total",
    "pipeline_model_param_memory_mb_total",
]
pipeline_wide = pipeline_df.pivot_table(
    index=["family", "model_variant", "window_size", "dataset"],
    columns="ratio",
    values=wide_metrics,
    aggfunc="first",
)
pipeline_wide.columns = [f"{m}_{r}" for m, r in pipeline_wide.columns]
pipeline_wide = pipeline_wide.reset_index()

pipe_wide_csv = TELEMETRY_DIR / f"pipeline_telemetry_wide_{TIMESTAMP}.csv"
pipe_wide_json = TELEMETRY_DIR / f"pipeline_telemetry_wide_{TIMESTAMP}.json"
pipeline_wide.to_csv(pipe_wide_csv, index=False)
pipeline_wide.to_json(pipe_wide_json, orient="records", indent=2)

all_csv = SUMMARY_DIR / f"bestfold_inference_all_ratios_{TIMESTAMP}.csv"
all_json = SUMMARY_DIR / f"bestfold_inference_all_ratios_{TIMESTAMP}.json"
pipeline_df.to_csv(all_csv, index=False)
pipeline_df.to_json(all_json, orient="records", indent=2)

for ratio_name in RATIOS:
    df_ratio = pipeline_df[pipeline_df["ratio"] == ratio_name].copy()
    if df_ratio.empty:
        continue
    ratio_csv = SUMMARY_DIR / f"bestfold_inference_ratio_{ratio_name}_{TIMESTAMP}.csv"
    ratio_json = SUMMARY_DIR / f"bestfold_inference_ratio_{ratio_name}_{TIMESTAMP}.json"
    df_ratio.to_csv(ratio_csv, index=False)
    df_ratio.to_json(ratio_json, orient="records", indent=2)

metric_keys = [
    k
    for k in pipeline_df.columns
    if (
        k.startswith("accuracy")
        or k.startswith("f1")
        or k.startswith("mcc")
        or k.startswith("balanced")
        or k.startswith("roc_auc")
        or k.startswith("pr_auc")
    )
]

def _row_to_eval_block(row_dict):
    return {
        "num_samples": int(row_dict["n_samples"]),
        "metrics": {k: row_dict[k] for k in metric_keys if k in row_dict},
        "confusion_matrix": json.loads(row_dict["confusion_matrix"]),
        "running_time_seconds": float(row_dict["mlp_running_time_seconds"]),
    }

row_gencode = pipeline_df[(pipeline_df["dataset"] == "gencode") & (pipeline_df["ratio"] == RAW_RATIO_TAG)].iloc[0].to_dict()
row_gtex = pipeline_df[(pipeline_df["dataset"] == "gtex") & (pipeline_df["ratio"] == RAW_RATIO_TAG)].iloc[0].to_dict()

legacy_inference_results[EXPERIMENT_NAME] = {
    "window_size": WINDOW_SIZE,
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "best_fold_number": 1,
    "checkpoint": best_checkpoint_name,
    "checkpoint_path": str(best_checkpoint_path),
    "model_inference_seconds_total": float(model_infer_total),
    "gencode_test": _row_to_eval_block(row_gencode),
    "gtex_test": _row_to_eval_block(row_gtex),
}

legacy_json = SUMMARY_DIR / f"best_fold_inference_results_{TIMESTAMP}.json"
with open(legacy_json, "w", encoding="utf-8") as f:
    json.dump(to_serializable(legacy_inference_results), f, indent=2)

print("\nInference + telemetry artifacts saved:")
print(f"  {mlp_long_csv}")
print(f"  {mlp_long_json}")
print(f"  {pipe_long_csv}")
print(f"  {pipe_long_json}")
print(f"  {pipe_wide_csv}")
print(f"  {pipe_wide_json}")
print(f"  {all_csv}")
print(f"  {all_json}")
print(f"  {legacy_json}")

print("\nQuick metric snapshot:")
print(
    pipeline_df[
        [
            "dataset",
            "ratio",
            "n_samples",
            "balanced_accuracy",
            "mcc",
            "f1_weighted",
            "roc_auc_macro",
            "pr_auc_Other",
            "pr_auc_Donor",
            "pr_auc_Acceptor",
        ]
    ].to_string(index=False)
)


Inference + telemetry artifacts saved:
  d:\Splice_FMs_seq_lengths\results\pipeline_telemetry\mlp_inference_telemetry_long_20260412_223542.csv
  d:\Splice_FMs_seq_lengths\results\pipeline_telemetry\mlp_inference_telemetry_long_20260412_223542.json
  d:\Splice_FMs_seq_lengths\results\pipeline_telemetry\pipeline_telemetry_long_20260412_223542.csv
  d:\Splice_FMs_seq_lengths\results\pipeline_telemetry\pipeline_telemetry_long_20260412_223542.json
  d:\Splice_FMs_seq_lengths\results\pipeline_telemetry\pipeline_telemetry_wide_20260412_223542.csv
  d:\Splice_FMs_seq_lengths\results\pipeline_telemetry\pipeline_telemetry_wide_20260412_223542.json
  d:\Splice_FMs_seq_lengths\results\fine_tuning_nt\summaries\bestfold_inference_all_ratios_20260412_223542.csv
  d:\Splice_FMs_seq_lengths\results\fine_tuning_nt\summaries\bestfold_inference_all_ratios_20260412_223542.json
  d:\Splice_FMs_seq_lengths\results\fine_tuning_nt\summaries\best_fold_inference_results_20260412_223542.json

Quick metric snapsh